# 06C — SFLLD Feature Engineering (Corrected Production Build)

## Purpose

This notebook is the **clean production replacement** for the earlier 06C feature-engineering build.  
The earlier notebook should be retained separately as a `vol(2)` validation/audit record; this file contains only the logic required to regenerate the production feature-engineered parquet.

### Governing leakage rule

\[
\boxed{\text{Train learns} \rightarrow \text{Freeze} \rightarrow \text{Validation applies}}
\]

Validation is never used to fit a scaler, K-means model, select \(K\), construct centroids, or determine semantic mappings.

## Corrections incorporated

1. **Macro feature order is frozen per specification.** Every scaler and K-means call uses the exact ordered list stored for that specification.
2. **No global/re-sorted feature list is allowed to transform a frozen model.** This prevents the feature-order error discovered during the MSA K=2 audit.
3. **Regular PIT and LS/Strict PIT remain independent FE pipelines.**
4. **State and MSA remain independent macro specifications.**
5. **Profile A and Profile B remain independent challengers.**
6. **Spatial \(K=8\) is retained as a previously validated design choice.**
7. **Scaling is re-audited at the end.** The notebook verifies scaler feature order, Train-only means/scales, standardized Train mean/variance, exact frozen-transform reproducibility, K-means dimensional compatibility, and non-degenerate assignments.
8. The production file is written to the same canonical path and therefore **overwrites the earlier `SFLLD_feature_engineered_train_validation.parquet` only after mandatory pre-save QA passes**.

### Important interpretation of scaling

Scaling is an **internal clustering transformation**. The parquet retains the original-scale macro variables plus cluster assignments; it does not replace macro variables with standardized columns. For each macro specification, `StandardScaler` is fit only on **unique complete Train macro profiles**, then frozen and applied in the same feature order to Train and Validation.


In [1]:
# 0. Imports and global configuration
from pathlib import Path
import json, math, warnings, hashlib
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import joblib

RANDOM_STATE = 42
SPATIAL_K = 8
MACRO_K_CANDIDATES = range(2, 7)
SILHOUETTE_SAMPLE = 20000

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

# Project root: notebook is expected under <PROJECT_ROOT>/notebooks.
# If run elsewhere, replace ROOT manually.
ROOT = Path.cwd().resolve()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent
elif not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

INPUT_PATH = ROOT / "data" / "output" / "final_modeling_dataset" / "SFLLD_feature_engineering_master_train_validation.parquet"
EXTERNAL_DIR = ROOT / "data" / "external"
GAZETTEER_PATH = EXTERNAL_DIR / "2020_Gaz_zcta_national.txt"
STATE_ZIP_PATH = EXTERNAL_DIR / "cb_2022_us_state_500k.zip"
STATE_ZIP_URL = "https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_us_state_500k.zip"

QA_DIR = ROOT / "data" / "output" / "quality_assurance" / "06C_feature_engineering"
ARTIFACT_DIR = QA_DIR / "artifacts"
MODEL_DIR = ARTIFACT_DIR / "models"
OUTPUT_PATH = ROOT / "data" / "output" / "final_modeling_dataset" / "SFLLD_feature_engineered_train_validation.parquet"
for p in [QA_DIR, ARTIFACT_DIR, MODEL_DIR, OUTPUT_PATH.parent]: p.mkdir(parents=True, exist_ok=True)

print("ROOT       :", ROOT)
print("INPUT      :", INPUT_PATH)
print("GAZETTEER  :", GAZETTEER_PATH)
print("STATE ZIP  :", STATE_ZIP_PATH)
print("OUTPUT     :", OUTPUT_PATH)
print("MODE       : corrected production rebuild; canonical parquet will be overwritten only after QA passes")
assert INPUT_PATH.exists(), f"Missing 06B master: {INPUT_PATH}"
assert GAZETTEER_PATH.exists(), f"Missing ZCTA gazetteer: {GAZETTEER_PATH}"
print("State boundary archive present:", STATE_ZIP_PATH.exists())
if not STATE_ZIP_PATH.exists():
    print("Official Census source (optional for this notebook's calculations):", STATE_ZIP_URL)


ROOT       : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6
INPUT      : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_feature_engineering_master_train_validation.parquet
GAZETTEER  : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\external\2020_Gaz_zcta_national.txt
STATE ZIP  : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\external\cb_2022_us_state_500k.zip
OUTPUT     : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_feature_engineered_train_validation.parquet
MODE       : corrected production rebuild; canonical parquet will be overwritten only after QA passes
State boundary archive present: True


1. Load 06B master and physically isolate Train/Validation
¶
All learned parameters are fit only on the Train object.
Target_24M
is retained for later supervised modeling/evaluation but is
never
passed to any FE learner.

In [2]:
df0 = pd.read_parquet(INPUT_PATH)
required = {"Loan_ID", "Split", "Target_24M"}
missing = required - set(df0.columns)
assert not missing, f"Required columns missing: {sorted(missing)}"

split_norm = df0["Split"].astype("string").str.strip()
assert set(split_norm.dropna().unique()) == {"Train", "Validation"}, split_norm.value_counts(dropna=False)
assert df0["Loan_ID"].notna().all(), "Loan_ID contains missing values."
assert df0["Loan_ID"].is_unique, "Loan_ID must be unique before FE."

_original_ids = df0["Loan_ID"].copy()
_original_order_hash = hashlib.sha256(pd.util.hash_pandas_object(_original_ids, index=False).values.tobytes()).hexdigest()
train = df0.loc[split_norm.eq("Train")].copy()
validation = df0.loc[split_norm.eq("Validation")].copy()

print(f"Input: {len(df0):,} rows × {df0.shape[1]} columns")
print(f"Train: {len(train):,} | Validation: {len(validation):,}")
display(split_norm.value_counts().rename_axis("Split").reset_index(name="N"))


Input: 209,041 rows × 66 columns
Train: 160,003 | Validation: 49,038


,Split,N
0,Train,160003
1,Validation,49038


2. Frozen 06A risk definitions
¶
These are implementation rules, not 06C tuning candidates. DTI is special: an originally missing DTI must become
Missing_DTI
, even if the processed numeric DTI was median-imputed.

In [3]:
# 2A. Helpers and frozen definitions
FICO_BINS = [-np.inf, 660, 700, 740, 760, np.inf]
FICO_LABELS = ["Very_High_Risk", "High_Risk", "Moderate_Risk", "Low_Risk", "Very_Low_Risk"]
DTI_BINS = [-np.inf, 25, 35, 40, np.inf]
DTI_LABELS = ["Low_DTI", "Moderate_DTI", "Elevated_DTI", "High_DTI"]
LEV_BINS = [-np.inf, 80, 90, 95, 100, np.inf]
LTV_LABELS = ["LTV_<80", "LTV_80_89", "LTV_90_94", "LTV_95_99", "LTV_100plus"]
CLTV_LABELS = ["CLTV_<80", "CLTV_80_89", "CLTV_90_94", "CLTV_95_99", "CLTV_100plus"]

# Optional cross-check against the saved 06A manifest if present.
manifest_path = QA_DIR.parent / "06A_risk_band_cutoff_discovery" / "06A_part2_frozen_risk_band_cutoffs.json"
print("06A cutoff manifest found:", manifest_path.exists())
if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        frozen_manifest = json.load(f)
    print("Loaded 06A manifest for provenance. The notebook's explicit final cutoffs remain authoritative for implementation.")

# Detect the processed DTI column. In this project Original_DTI is the retained processed numeric field,
# while DTI_Missing preserves pre-imputation missingness.
assert "Original_DTI" in df0.columns and "DTI_Missing" in df0.columns

def apply_risk_features(d):
    d = d.copy()
    d["FICO_Risk_Band"] = pd.cut(d["FICO_Score"], FICO_BINS, labels=FICO_LABELS, right=False, ordered=True)
    observed_band = pd.cut(d["Original_DTI"], DTI_BINS, labels=DTI_LABELS, right=False, ordered=True).astype("string")
    dti_missing = pd.to_numeric(d["DTI_Missing"], errors="coerce").fillna(0).astype(int).eq(1)
    d["DTI_Risk_Band"] = observed_band.mask(dti_missing, "Missing_DTI")
    d["DTI_Risk_Band"] = pd.Categorical(d["DTI_Risk_Band"], categories=["Missing_DTI"] + DTI_LABELS, ordered=True)
    d["LTV_Risk_Component"] = pd.cut(d["Original_LTV"], LEV_BINS, labels=LTV_LABELS, right=False, ordered=True)
    d["CLTV_Risk_Component"] = pd.cut(d["Original_CLTV"], LEV_BINS, labels=CLTV_LABELS, right=False, ordered=True)
    return d

train = apply_risk_features(train)
validation = apply_risk_features(validation)
print("Frozen risk-band features created.")


06A cutoff manifest found: True
Loaded 06A manifest for provenance. The notebook's explicit final cutoffs remain authoritative for implementation.
Frozen risk-band features created.


3. Deterministic mortgage features
¶
No distributional statistic is learned here. The same formula is applied to both splits.
Feature
Formula / rule
Unit
CLTV_LTV_Gap
CLTV − LTV
percentage points
DTI_LTV_Interaction
DTI × LTV
%²
DTI_Rate_Interaction
DTI × note rate
%²
UPB_per_Borrower
UPB / borrowers
USD/borrower
Estimated_Monthly_PI
standard amortizing P&I
USD/month
High_Leverage
1 if LTV ≥100 or CLTV ≥100
binary
HARP_High_Leverage
HARP × High Leverage
binary
FICO_DTI_Interaction
FICO × DTI
score×%
FICO_LTV_Interaction
FICO × LTV
score×%
DTI-derived numeric interactions are set to
NaN
when
DTI_Missing=1
; this prevents a median-imputed DTI from masquerading as observed borrower information. The explicit
DTI_Missing
and
DTI_Risk_Band=Missing_DTI
features preserve that mechanism.

In [4]:
def apply_deterministic_features(d):
    d = d.copy()
    dti_missing = pd.to_numeric(d["DTI_Missing"], errors="coerce").fillna(0).astype(int).eq(1)
    dti_obs = pd.to_numeric(d["Original_DTI"], errors="coerce").mask(dti_missing)
    ltv = pd.to_numeric(d["Original_LTV"], errors="coerce")
    cltv = pd.to_numeric(d["Original_CLTV"], errors="coerce")
    fico = pd.to_numeric(d["FICO_Score"], errors="coerce")
    rate_pct = pd.to_numeric(d["Original_Interest_Rate"], errors="coerce")
    upb = pd.to_numeric(d["Original_UPB"], errors="coerce")
    borrowers = pd.to_numeric(d["Number_of_Borrowers"], errors="coerce")
    term_months = pd.to_numeric(d["Original_Loan_Term"], errors="coerce")

    d["CLTV_LTV_Gap"] = cltv - ltv
    d["DTI_LTV_Interaction"] = dti_obs * ltv
    d["DTI_Rate_Interaction"] = dti_obs * rate_pct
    d["UPB_per_Borrower"] = upb / borrowers.where(borrowers > 0)

    i = (rate_pct / 100.0) / 12.0
    n = term_months
    pi = pd.Series(np.nan, index=d.index, dtype="float64")
    valid = upb.notna() & i.notna() & n.gt(0)
    zero = valid & np.isclose(i, 0.0)
    nonzero = valid & ~zero
    pi.loc[zero] = upb.loc[zero] / n.loc[zero]
    base = 1.0 + i.loc[nonzero]
    pi.loc[nonzero] = upb.loc[nonzero] * (i.loc[nonzero] * np.power(base, n.loc[nonzero])) / (np.power(base, n.loc[nonzero]) - 1.0)
    d["Estimated_Monthly_PI"] = pi

    d["High_Leverage"] = ((ltv >= 100) | (cltv >= 100)).astype("int8")
    harp = d["HARP_Indicator"].astype("string").str.strip().str.upper().eq("Y")
    d["HARP_High_Leverage"] = (harp & d["High_Leverage"].eq(1)).astype("int8")
    d["FICO_DTI_Interaction"] = fico * dti_obs
    d["FICO_LTV_Interaction"] = fico * ltv
    return d

train = apply_deterministic_features(train)
validation = apply_deterministic_features(validation)
print("Deterministic mortgage FE complete.")


Deterministic mortgage FE complete.


4. Spatial feature engineering
¶
The Census ZCTA Gazetteer supplies representative latitude/longitude. Coordinates are converted to 3D unit-sphere vectors to avoid treating longitude as a flat Euclidean axis:
[
x=\cos\phi\cos\lambda,\quad y=\cos\phi\sin\lambda,\quad z=\sin\phi
]
The
Train loan population
fits the 8-cluster K-means, then the fitted centroids are frozen and used to assign Validation. The Census state boundary ZIP is retained as a project reference/provenance input; the clustering calculation itself does not require polygon geometry.

In [5]:
# 4A. Read 2020 ZCTA Gazetteer robustly
# Census Gazetteer files are tab-delimited; column names commonly include GEOID, INTPTLAT, INTPTLONG.
gaz = pd.read_csv(GAZETTEER_PATH, sep="\t", dtype="string")
gaz.columns = [str(c).strip() for c in gaz.columns]
print("Gazetteer columns:", gaz.columns.tolist())

def pick_col(cols, candidates):
    norm = {c.upper().strip(): c for c in cols}
    for x in candidates:
        if x.upper() in norm: return norm[x.upper()]
    raise KeyError(f"None of {candidates} found in columns: {list(cols)}")

zcta_col = pick_col(gaz.columns, ["GEOID", "GEOID20", "ZCTA5CE20"])
lat_col = pick_col(gaz.columns, ["INTPTLAT", "INTPTLAT20", "INTPTLAT10"])
lon_col = pick_col(gaz.columns, ["INTPTLONG", "INTPTLON", "INTPTLONG20", "INTPTLON20"])

gaz_ref = pd.DataFrame({
    "ZCTA5": gaz[zcta_col].str.extract(r"(\d{5})", expand=False).str.zfill(5),
    "Latitude": pd.to_numeric(gaz[lat_col], errors="coerce"),
    "Longitude": pd.to_numeric(gaz[lon_col], errors="coerce"),
}).dropna().drop_duplicates("ZCTA5")
gaz_ref["ZIP3"] = gaz_ref["ZCTA5"].str[:3]

# One coordinate per ZIP3: unweighted mean of ZCTA representative points.
zip3_ref = gaz_ref.groupby("ZIP3", as_index=False).agg(Latitude=("Latitude", "mean"), Longitude=("Longitude", "mean"), ZCTA_Count=("ZCTA5", "nunique"))
print(f"ZCTA rows: {len(gaz_ref):,}; ZIP3 reference rows: {len(zip3_ref):,}")
display(zip3_ref.head())


Gazetteer columns: ['GEOID', 'ALAND', 'AWATER', 'ALAND_SQMI', 'AWATER_SQMI', 'INTPTLAT', 'INTPTLONG']
ZCTA rows: 33,144; ZIP3 reference rows: 896


,ZIP3,Latitude,Longitude,ZCTA_Count
0,006,18.274107,-66.831553,44
1,007,18.158462,-66.113778,46
2,008,17.956453,-64.795246,11
3,009,18.408407,-66.092961,41
4,010,42.267524,-72.571616,63


In [6]:
# 4B. Normalize Postal_Code to ZIP3 and attach coordinates
# Missing-like MSA is unrelated; spatial geography is driven by Postal_Code/ZIP3.
def to_zip3(s):
    x = s.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    digits = x.str.extract(r"(\d+)", expand=False)
    # Freddie Postal_Code is commonly ZIP3 already. Preserve first three digits.
    return digits.str.zfill(3).str[:3]

zip3_lookup = zip3_ref.set_index("ZIP3")[["Latitude", "Longitude"]]

def attach_spatial_coords(d):
    d = d.copy()
    d["_ZIP3_FE"] = to_zip3(d["Postal_Code"])
    d["Spatial_Latitude"] = d["_ZIP3_FE"].map(zip3_lookup["Latitude"])
    d["Spatial_Longitude"] = d["_ZIP3_FE"].map(zip3_lookup["Longitude"])
    lat = np.deg2rad(d["Spatial_Latitude"])
    lon = np.deg2rad(d["Spatial_Longitude"])
    d["_Spatial_X"] = np.cos(lat) * np.cos(lon)
    d["_Spatial_Y"] = np.cos(lat) * np.sin(lon)
    d["_Spatial_Z"] = np.sin(lat)
    return d

train = attach_spatial_coords(train)
validation = attach_spatial_coords(validation)
spatial_cov = pd.DataFrame({
    "Split": ["Train", "Validation"],
    "N": [len(train), len(validation)],
    "Coordinate_Missing_N": [train["_Spatial_X"].isna().sum(), validation["_Spatial_X"].isna().sum()]
})
spatial_cov["Coordinate_Missing_Pct"] = spatial_cov["Coordinate_Missing_N"] / spatial_cov["N"] * 100
display(spatial_cov)
spatial_cov.to_csv(QA_DIR / "06C_spatial_coordinate_coverage.csv", index=False)


,Split,N,Coordinate_Missing_N,Coordinate_Missing_Pct
0,Train,160003,0,0.0
1,Validation,49038,0,0.0


In [7]:
# 4C. Fit on Train only, predict Validation only
spatial_features = ["_Spatial_X", "_Spatial_Y", "_Spatial_Z"]
train_spatial_ok = train[spatial_features].notna().all(axis=1)
val_spatial_ok = validation[spatial_features].notna().all(axis=1)
assert train_spatial_ok.sum() >= SPATIAL_K

spatial_kmeans = KMeans(n_clusters=SPATIAL_K, random_state=RANDOM_STATE, n_init=20)
spatial_kmeans.fit(train.loc[train_spatial_ok, spatial_features])

train["Spatial_Cluster"] = pd.Series(pd.NA, index=train.index, dtype="Int64")
validation["Spatial_Cluster"] = pd.Series(pd.NA, index=validation.index, dtype="Int64")
train.loc[train_spatial_ok, "Spatial_Cluster"] = spatial_kmeans.predict(train.loc[train_spatial_ok, spatial_features])
validation.loc[val_spatial_ok, "Spatial_Cluster"] = spatial_kmeans.predict(validation.loc[val_spatial_ok, spatial_features])

# Frozen interpretation mapping from the prior spatial-cluster analysis.
SPATIAL_REGION_MAP = {
    0: "Central Plains", 1: "Southeast", 2: "Southwest & California", 3: "Pacific Northwest",
    4: "Midwest", 5: "Northeast & Mid-Atlantic", 6: "South-Central", 7: "Pacific Islands"
}
train["Spatial_Cluster_Region"] = train["Spatial_Cluster"].map(SPATIAL_REGION_MAP).astype("string")
validation["Spatial_Cluster_Region"] = validation["Spatial_Cluster"].map(SPATIAL_REGION_MAP).astype("string")

joblib.dump(spatial_kmeans, MODEL_DIR / "06C_spatial_kmeans_train_only.joblib")
with open(ARTIFACT_DIR / "06C_spatial_cluster_region_mapping.json", "w", encoding="utf-8") as f:
    json.dump(SPATIAL_REGION_MAP, f, indent=2)

spatial_summary = pd.concat([
    train.assign(_Split="Train"), validation.assign(_Split="Validation")
]).groupby(["_Split", "Spatial_Cluster"], dropna=False).size().rename("N").reset_index()
display(spatial_summary)
spatial_summary.to_csv(QA_DIR / "06C_spatial_cluster_distribution.csv", index=False)


,_Split,Spatial_Cluster,N
0,Train,0,30311
1,Train,1,32023
2,Train,2,9711
3,Train,3,25911
4,Train,4,35684
5,Train,5,9728
6,Train,6,652
7,Train,7,15983
8,Validation,0,8407
9,Validation,1,9345


5. Macro regime FE — 8 independent candidate specifications
¶
We deliberately keep separate:
Regular PIT vs LS/Strict PIT;
State vs MSA HPI geography;
Profile A vs Profile B.
Profile A:
[HPI, HPI_Growth_12M, UR, Employment_Growth_12M, ΔUR_12M]
Profile B:
[HPI_Growth_12M, UR, Employment_Growth_12M, ΔUR_12M]
Each model constructs
unique Train macro vectors
, fits its own
StandardScaler
, evaluates candidate K on Train only, freezes the chosen scaler/K-means, and applies them to Validation. Rows lacking a complete defensible vector receive an explicit unresolved cluster category (
-1
) rather than median fabrication.
Because the retained BLS variables are resolved/final fields, the output names explicitly contain
ResolvedBLS
. This is intentional provenance and prevents overclaiming geography purity.

In [8]:
# 5A. Resolve retained macro column names
REG_BLS = ["BLS_Unemployment_Rate_Final", "BLS_Employment_Growth_12M_Final", "BLS_Unemployment_Rate_Change_12M_Final"]
LS_BLS  = ["LS_BLS_Unemployment_Rate_Final", "LS_BLS_Employment_Growth_12M_Final", "LS_BLS_Unemployment_Rate_Change_12M_Final"]
for c in REG_BLS + LS_BLS:
    assert c in df0.columns, f"Required retained macro field missing: {c}"

MACRO_SPECS = {
    "Regular_State_A": ["FHFA_State_HPI", "FHFA_State_HPI_Growth_12M"] + REG_BLS,
    "Regular_State_B": ["FHFA_State_HPI_Growth_12M"] + REG_BLS,
    "LS_State_A": ["LS_FHFA_State_HPI", "LS_FHFA_State_HPI_Growth_12M"] + LS_BLS,
    "LS_State_B": ["LS_FHFA_State_HPI_Growth_12M"] + LS_BLS,
    "Regular_MSA_A": ["FHFA_MSA_HPI", "FHFA_MSA_HPI_Growth_12M"] + REG_BLS,
    "Regular_MSA_B": ["FHFA_MSA_HPI_Growth_12M"] + REG_BLS,
    "LS_MSA_A": ["LS_FHFA_MSA_HPI", "LS_FHFA_MSA_HPI_Growth_12M"] + LS_BLS,
    "LS_MSA_B": ["LS_FHFA_MSA_HPI_Growth_12M"] + LS_BLS,
}
for name, cols in MACRO_SPECS.items():
    miss = [c for c in cols if c not in df0.columns]
    assert not miss, f"{name}: missing {miss}"

macro_manifest = pd.DataFrame([{"Specification": k, "Columns": " | ".join(v), "N_Features": len(v)} for k,v in MACRO_SPECS.items()])
display(macro_manifest)
macro_manifest.to_csv(QA_DIR / "06C_macro_profile_manifest.csv", index=False)


,Specification,Columns,N_Features
0,Regular_State_A,FHFA_State_HPI | FHFA_State_HPI_Growth_12M | B...,5
1,Regular_State_B,FHFA_State_HPI_Growth_12M | BLS_Unemployment_R...,4
2,LS_State_A,LS_FHFA_State_HPI | LS_FHFA_State_HPI_Growth_1...,5
3,LS_State_B,LS_FHFA_State_HPI_Growth_12M | LS_BLS_Unemploy...,4
4,Regular_MSA_A,FHFA_MSA_HPI | FHFA_MSA_HPI_Growth_12M | BLS_U...,5
5,Regular_MSA_B,FHFA_MSA_HPI_Growth_12M | BLS_Unemployment_Rat...,4
6,LS_MSA_A,LS_FHFA_MSA_HPI | LS_FHFA_MSA_HPI_Growth_12M |...,5
7,LS_MSA_B,LS_FHFA_MSA_HPI_Growth_12M | LS_BLS_Unemployme...,4


In [9]:
# 5B. Train-only macro scaling, K selection, frozen transform
# CORRECTED PRODUCTION IMPLEMENTATION
#
# Key correction:
#   Every fitted bundle stores its own ordered feature list and every subsequent
#   scaler/KMeans transform uses ONLY that exact ordered list.
#
# No SPEC_FEATURES/global set/sorted list is allowed to transform a frozen model.

def fit_macro_spec(train_df, val_df, spec_name, features):
    frozen_features = list(features)  # preserve declared order exactly

    assert len(frozen_features) == len(set(frozen_features)), (
        f"{spec_name}: duplicate feature names in specification."
    )
    missing_cols = [c for c in frozen_features if c not in train_df.columns or c not in val_df.columns]
    if missing_cols:
        raise KeyError(f"{spec_name}: missing macro columns: {missing_cols}")

    tr_ok = train_df[frozen_features].notna().all(axis=1)
    va_ok = val_df[frozen_features].notna().all(axis=1)

    # Unique macro vectors prevent repeated loans from overweighting identical macro states.
    X_unique = (
        train_df.loc[tr_ok, frozen_features]
        .drop_duplicates()
        .astype(float)
        .reset_index(drop=True)
    )
    if len(X_unique) < 3:
        raise ValueError(
            f"{spec_name}: insufficient complete unique Train macro profiles ({len(X_unique)})."
        )

    # Train-only scaler.
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_unique[frozen_features])

    # Explicit structural checks immediately after fitting.
    assert scaler.n_features_in_ == len(frozen_features)
    if hasattr(scaler, "feature_names_in_"):
        assert list(scaler.feature_names_in_) == frozen_features, (
            f"{spec_name}: scaler feature order differs from frozen feature order."
        )
    assert Xs.shape == X_unique.shape
    assert np.isfinite(Xs).all()

    diagnostics = []
    valid_ks = [k for k in MACRO_K_CANDIDATES if k < len(X_unique)]

    for k in valid_ks:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
        labels = km.fit_predict(Xs)
        counts = np.bincount(labels, minlength=k)

        # Deterministic sample for computationally safe silhouette.
        if len(Xs) > SILHOUETTE_SAMPLE:
            rng = np.random.default_rng(RANDOM_STATE)
            idx = np.sort(rng.choice(len(Xs), size=SILHOUETTE_SAMPLE, replace=False))
            sil = silhouette_score(Xs[idx], labels[idx])
        else:
            sil = silhouette_score(Xs, labels)

        diagnostics.append({
            "Specification": spec_name,
            "K": k,
            "Inertia": km.inertia_,
            "Silhouette": sil,
            "Min_Cluster_N": int(counts.min()),
            "Max_Cluster_N": int(counts.max()),
            "Unique_Train_Profiles": len(X_unique),
        })

    diag = pd.DataFrame(diagnostics)

    # Production rule retained from the original 06C build:
    # maximize Train-only silhouette; tie-break toward smaller K.
    # The separate vol(2) robustness/economic-interpretation notebook remains the
    # place for K=2..6 challenger analysis; Validation is never used here.
    chosen_k = int(
        diag.sort_values(["Silhouette", "K"], ascending=[False, True])
        .iloc[0]["K"]
    )

    model = KMeans(
        n_clusters=chosen_k,
        random_state=RANDOM_STATE,
        n_init=20,
    ).fit(Xs)

    assert model.cluster_centers_.shape[1] == len(frozen_features)

    out_col = f"{spec_name}_Macro_Cluster_ResolvedBLS"
    tr_labels = pd.Series(-1, index=train_df.index, dtype="Int64")
    va_labels = pd.Series(-1, index=val_df.index, dtype="Int64")

    # IMPORTANT: use the frozen feature order explicitly on both splits.
    Xtr = train_df.loc[tr_ok, frozen_features].astype(float)
    Xva = val_df.loc[va_ok, frozen_features].astype(float)

    Xtr_s = scaler.transform(Xtr[frozen_features])
    Xva_s = scaler.transform(Xva[frozen_features])

    tr_labels.loc[tr_ok] = model.predict(Xtr_s)
    va_labels.loc[va_ok] = model.predict(Xva_s)

    # Centroids returned to original units for interpretation.
    cent_orig = scaler.inverse_transform(model.cluster_centers_)
    cent = pd.DataFrame(cent_orig, columns=frozen_features)
    cent.insert(0, "Cluster", np.arange(chosen_k))
    cent.insert(0, "Specification", spec_name)

    coverage = pd.DataFrame([
        {
            "Specification": spec_name,
            "Split": "Train",
            "N": len(train_df),
            "Complete_Vector_N": int(tr_ok.sum()),
            "Unresolved_N": int((~tr_ok).sum()),
        },
        {
            "Specification": spec_name,
            "Split": "Validation",
            "N": len(val_df),
            "Complete_Vector_N": int(va_ok.sum()),
            "Unresolved_N": int((~va_ok).sum()),
        },
    ])
    coverage["Complete_Vector_Pct"] = coverage["Complete_Vector_N"] / coverage["N"] * 100
    coverage["Unresolved_Pct"] = coverage["Unresolved_N"] / coverage["N"] * 100

    bundle = {
        "features": frozen_features,
        "scaler": scaler,
        "kmeans": model,
        "chosen_k": chosen_k,
        "random_state": RANDOM_STATE,
        "training_unique_profiles": len(X_unique),
        "labor_semantics": "retained resolved/final PIT BLS fields",
    }

    joblib.dump(
        bundle,
        MODEL_DIR / f"06C_{spec_name}_resolvedBLS_train_only.joblib",
    )

    return out_col, tr_labels, va_labels, diag, cent, coverage, bundle


macro_diags, macro_centroids, macro_coverage = [], [], []
model_store = {}

for spec_name, features in MACRO_SPECS.items():
    out_col, tr_lab, va_lab, diag, cent, cov, bundle = fit_macro_spec(
        train, validation, spec_name, features
    )
    train[out_col] = tr_lab
    validation[out_col] = va_lab

    macro_diags.append(diag)
    macro_centroids.append(cent)
    macro_coverage.append(cov)

    # Canonical frozen bundle used by all later QA.
    model_store[(spec_name, int(bundle["chosen_k"]))] = bundle

macro_diags = pd.concat(macro_diags, ignore_index=True)
macro_centroids = pd.concat(macro_centroids, ignore_index=True)
macro_coverage = pd.concat(macro_coverage, ignore_index=True)

macro_diags.to_csv(QA_DIR / "06C_macro_k_diagnostics_train_only.csv", index=False)
macro_centroids.to_csv(QA_DIR / "06C_macro_centroids_original_units.csv", index=False)
macro_coverage.to_csv(QA_DIR / "06C_macro_cluster_coverage.csv", index=False)

display(macro_diags)
display(macro_coverage)

print("\nFrozen production macro bundles:")
for (spec, k), bundle in model_store.items():
    print(f"  {spec}: K={k}, features={bundle['features']}")


,Specification,K,Inertia,Silhouette,Min_Cluster_N,Max_Cluster_N,Unique_Train_Profiles
0,Regular_State_A,2,63791.725040,0.266792,4979,11011,15990
1,Regular_State_A,3,55668.600426,0.164935,4217,6036,15990
2,Regular_State_A,4,49671.115169,0.180384,1694,5809,15990
3,Regular_State_A,5,44156.147176,0.186207,1304,4825,15990
4,Regular_State_A,6,40120.516429,0.195149,525,5425,15990
5,Regular_State_B,2,49605.654021,0.261293,5699,10291,15990
6,Regular_State_B,3,42371.386790,0.199036,4306,6312,15990
7,Regular_State_B,4,36125.073372,0.209238,2123,4873,15990
8,Regular_State_B,5,32385.321375,0.213027,602,5611,15990
9,Regular_State_B,6,29142.243205,0.206137,50,5519,15990


,Specification,Split,N,Complete_Vector_N,Unresolved_N,Complete_Vector_Pct,Unresolved_Pct
0,Regular_State_A,Train,160003,159917,86,99.946251,0.053749
1,Regular_State_A,Validation,49038,49025,13,99.973490,0.026510
2,Regular_State_B,Train,160003,159917,86,99.946251,0.053749
3,Regular_State_B,Validation,49038,49025,13,99.973490,0.026510
4,LS_State_A,Train,160003,159917,86,99.946251,0.053749
5,LS_State_A,Validation,49038,49025,13,99.973490,0.026510
6,LS_State_B,Train,160003,159917,86,99.946251,0.053749
7,LS_State_B,Validation,49038,49025,13,99.973490,0.026510
8,Regular_MSA_A,Train,160003,128197,31806,80.121623,19.878377
9,Regular_MSA_A,Validation,49038,40450,8588,82.487051,17.512949



Frozen production macro bundles:
  Regular_State_A: K=2, features=['FHFA_State_HPI', 'FHFA_State_HPI_Growth_12M', 'BLS_Unemployment_Rate_Final', 'BLS_Employment_Growth_12M_Final', 'BLS_Unemployment_Rate_Change_12M_Final']
  Regular_State_B: K=2, features=['FHFA_State_HPI_Growth_12M', 'BLS_Unemployment_Rate_Final', 'BLS_Employment_Growth_12M_Final', 'BLS_Unemployment_Rate_Change_12M_Final']
  LS_State_A: K=2, features=['LS_FHFA_State_HPI', 'LS_FHFA_State_HPI_Growth_12M', 'LS_BLS_Unemployment_Rate_Final', 'LS_BLS_Employment_Growth_12M_Final', 'LS_BLS_Unemployment_Rate_Change_12M_Final']
  LS_State_B: K=2, features=['LS_FHFA_State_HPI_Growth_12M', 'LS_BLS_Unemployment_Rate_Final', 'LS_BLS_Employment_Growth_12M_Final', 'LS_BLS_Unemployment_Rate_Change_12M_Final']
  Regular_MSA_A: K=2, features=['FHFA_MSA_HPI', 'FHFA_MSA_HPI_Growth_12M', 'BLS_Unemployment_Rate_Final', 'BLS_Employment_Growth_12M_Final', 'BLS_Unemployment_Rate_Change_12M_Final']
  Regular_MSA_B: K=2, features=['FHFA_MSA_HPI_

6. Recombine splits and remove construction-only helper columns
¶
The original row order and Loan_ID identity are restored explicitly. Learned artifacts remain saved separately.

In [10]:
train["_Original_Row_Order"] = train.index
validation["_Original_Row_Order"] = validation.index
fe = pd.concat([train, validation], axis=0).sort_values("_Original_Row_Order").drop(columns="_Original_Row_Order")

helper_cols = ["_ZIP3_FE", "_Spatial_X", "_Spatial_Y", "_Spatial_Z"]
fe = fe.drop(columns=[c for c in helper_cols if c in fe.columns])

assert len(fe) == len(df0)
assert fe["Loan_ID"].is_unique
assert fe["Loan_ID"].reset_index(drop=True).equals(df0["Loan_ID"].reset_index(drop=True))
print(f"Recombined FE dataset: {len(fe):,} rows × {fe.shape[1]} columns")


Recombined FE dataset: 209,041 rows × 91 columns


7. Mandatory post-FE validation
¶
06C is not complete merely because feature columns were created. The following checks test whether the output matches the intended definitions and whether leakage/identity errors occurred.

In [11]:
qa = []
def check(name, passed, detail=""):
    qa.append({"Check": name, "Passed": bool(passed), "Detail": str(detail)})

# 7.1 Identity / split / target preservation
check("Row count unchanged", len(fe) == len(df0), f"{len(df0)} -> {len(fe)}")
check("Loan_ID unique", fe["Loan_ID"].is_unique)
check("Loan_ID order unchanged", fe["Loan_ID"].reset_index(drop=True).equals(df0["Loan_ID"].reset_index(drop=True)))
check("Split unchanged", fe["Split"].astype("string").reset_index(drop=True).equals(df0["Split"].astype("string").reset_index(drop=True)))
check("Target unchanged", fe["Target_24M"].reset_index(drop=True).equals(df0["Target_24M"].reset_index(drop=True)))

# 7.2 DTI missingness rule
_dti_m = pd.to_numeric(fe["DTI_Missing"], errors="coerce").fillna(0).astype(int).eq(1)
check("All DTI_Missing=1 -> Missing_DTI", fe.loc[_dti_m, "DTI_Risk_Band"].astype("string").eq("Missing_DTI").all(), f"N={_dti_m.sum()}")
check("No observed DTI -> Missing_DTI", ~fe.loc[~_dti_m, "DTI_Risk_Band"].astype("string").eq("Missing_DTI").any())
check("DTI interactions suppressed when originally missing", fe.loc[_dti_m, ["DTI_LTV_Interaction","DTI_Rate_Interaction","FICO_DTI_Interaction"]].isna().all().all())

# 7.3 Deterministic formula checks
check("CLTV_LTV_Gap formula", np.allclose(fe["CLTV_LTV_Gap"], pd.to_numeric(fe["Original_CLTV"],errors="coerce")-pd.to_numeric(fe["Original_LTV"],errors="coerce"), equal_nan=True))
expected_high = ((pd.to_numeric(fe["Original_LTV"],errors="coerce")>=100)|(pd.to_numeric(fe["Original_CLTV"],errors="coerce")>=100)).astype("int8")
check("High_Leverage formula", fe["High_Leverage"].equals(expected_high))

# 7.4 Spatial checks
check("Spatial cluster IDs valid", fe["Spatial_Cluster"].dropna().between(0, SPATIAL_K-1).all())
check("Spatial labels agree with frozen map", (fe.loc[fe["Spatial_Cluster"].notna(), "Spatial_Cluster_Region"].astype("string") == fe.loc[fe["Spatial_Cluster"].notna(), "Spatial_Cluster"].map(SPATIAL_REGION_MAP).astype("string")).all())

# 7.5 Macro output checks: -1 is the explicit unresolved category.
macro_cols = [c for c in fe.columns if c.endswith("_Macro_Cluster_ResolvedBLS")]
check("All 8 macro candidate columns created", len(macro_cols) == 8, macro_cols)
for c in macro_cols:
    check(f"{c}: no pandas NaN", fe[c].isna().sum() == 0, f"-1 unresolved N={(fe[c]==-1).sum()}")

qa_table = pd.DataFrame(qa)
display(qa_table)
qa_table.to_csv(QA_DIR / "06C_quality_assurance_checks.csv", index=False)

failed = qa_table.loc[~qa_table["Passed"]]
if len(failed):
    display(failed)
    raise AssertionError(f"06C QA FAILED: {len(failed)} check(s) failed. Review table above.")
print("ALL MANDATORY 06C QA CHECKS PASSED.")


C:\Users\itwill\AppData\Local\Temp\ipykernel_19964\3609480908.py:15: DeprecationWarning: Bitwise inversion '~' on bool is deprecated and will be removed in Python 3.16. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  check("No observed DTI -> Missing_DTI", ~fe.loc[~_dti_m, "DTI_Risk_Band"].astype("string").eq("Missing_DTI").any())


,Check,Passed,Detail
0,Row count unchanged,True,209041 -> 209041
1,Loan_ID unique,True,
2,Loan_ID order unchanged,True,
3,Split unchanged,True,
4,Target unchanged,True,
5,All DTI_Missing=1 -> Missing_DTI,True,N=17595
6,No observed DTI -> Missing_DTI,True,
7,DTI interactions suppressed when originally mi...,True,
8,CLTV_LTV_Gap formula,True,
9,High_Leverage formula,True,


ALL MANDATORY 06C QA CHECKS PASSED.


In [12]:
# 7.6 New-feature missingness and split distributions
new_cols = [c for c in fe.columns if c not in df0.columns]
records = []
for c in new_cols:
    for split in ["Train", "Validation"]:
        s = fe.loc[fe["Split"].astype("string").eq(split), c]
        records.append({"Column": c, "Split": split, "N": len(s), "Missing_N": int(s.isna().sum()), "Missing_Pct": float(s.isna().mean()*100), "Dtype": str(s.dtype)})
new_missing = pd.DataFrame(records).sort_values(["Missing_Pct","Column"], ascending=[False,True])
display(new_missing)
new_missing.to_csv(QA_DIR / "06C_new_feature_missingness_by_split.csv", index=False)

# Categorical/binary distributions for sanity review
review_cols = ["FICO_Risk_Band","DTI_Risk_Band","LTV_Risk_Component","CLTV_Risk_Component","High_Leverage","HARP_High_Leverage","Spatial_Cluster","Spatial_Cluster_Region"] + macro_cols
review = []
for c in review_cols:
    tmp = fe.groupby(["Split", c], observed=False, dropna=False).size().rename("N").reset_index()
    tmp.insert(0,"Feature",c)
    tmp = tmp.rename(columns={c:"Value"})
    review.append(tmp)
review = pd.concat(review, ignore_index=True)
display(review.head(100))
review.to_csv(QA_DIR / "06C_engineered_feature_distribution_by_split.csv", index=False)


,Column,Split,N,Missing_N,Missing_Pct,Dtype
10,DTI_LTV_Interaction,Train,160003,16001,10.000437,float64
12,DTI_Rate_Interaction,Train,160003,16001,10.000437,float64
22,FICO_DTI_Interaction,Train,160003,16001,10.000437,float64
11,DTI_LTV_Interaction,Validation,49038,1594,3.250540,float64
13,DTI_Rate_Interaction,Validation,49038,1594,3.250540,float64
23,FICO_DTI_Interaction,Validation,49038,1594,3.250540,float64
8,CLTV_LTV_Gap,Train,160003,0,0.000000,float64
9,CLTV_LTV_Gap,Validation,49038,0,0.000000,float64
6,CLTV_Risk_Component,Train,160003,0,0.000000,category
7,CLTV_Risk_Component,Validation,49038,0,0.000000,category


,Feature,Split,Value,N
0,FICO_Risk_Band,Train,Very_High_Risk,7831
1,FICO_Risk_Band,Train,High_Risk,19659
2,FICO_Risk_Band,Train,Moderate_Risk,33813
3,FICO_Risk_Band,Train,Low_Risk,21025
4,FICO_Risk_Band,Train,Very_Low_Risk,77675
...,...,...,...,...
95,LS_State_A_Macro_Cluster_ResolvedBLS,Validation,-1,13
96,LS_State_A_Macro_Cluster_ResolvedBLS,Validation,0,29314
97,LS_State_A_Macro_Cluster_ResolvedBLS,Validation,1,19711
98,LS_State_B_Macro_Cluster_ResolvedBLS,Train,-1,86


## 8. Corrected scaling-structure and frozen-transform audit

This is the mandatory audit added after the MSA feature-order investigation.

For every one of the eight macro specifications, this section verifies:

- the scaler was fit on the specification's **ordered Train feature vector**;
- `scaler.mean_` and `scaler.scale_` exactly match the unique complete Train macro profiles used for fitting;
- transformed Train fitting profiles are approximately centered at 0 with unit population variance;
- `scaler.transform()` is numerically identical to manual \((x-\mu)/\sigma\) transformation in the same feature order;
- K-means centroid dimensionality equals the frozen feature count;
- production cluster labels can be exactly reproduced from the frozen scaler/K-means bundle;
- Validation is transformed only with frozen Train parameters;
- Regular PIT and LS/Strict PIT have separate scaler and K-means objects.

**Important:** Validation is *not expected* to have standardized mean 0 or standard deviation 1. A Validation shift is an empirical distribution shift, not a scaler failure.


In [13]:
# 8A. Scaling / feature-order / frozen-transform integrity audit

scaling_rows = []
prediction_rows = []

for spec_name, declared_features in MACRO_SPECS.items():
    # Locate the single frozen production bundle for this specification.
    keys = [key for key in model_store if key[0] == spec_name]
    assert len(keys) == 1, f"{spec_name}: expected exactly one frozen production bundle, found {keys}"

    key = keys[0]
    bundle = model_store[key]
    frozen_features = list(bundle["features"])
    scaler = bundle["scaler"]
    kmeans = bundle["kmeans"]
    chosen_k = int(bundle["chosen_k"])

    # 1) Feature-order integrity.
    declared_order_ok = frozen_features == list(declared_features)
    scaler_order_ok = (
        list(scaler.feature_names_in_) == frozen_features
        if hasattr(scaler, "feature_names_in_")
        else True
    )
    dimension_ok = (
        scaler.n_features_in_ == len(frozen_features)
        and kmeans.cluster_centers_.shape[1] == len(frozen_features)
    )

    # 2) Reconstruct EXACT scaler fitting matrix: unique complete Train profiles.
    tr_ok = train[frozen_features].notna().all(axis=1)
    va_ok = validation[frozen_features].notna().all(axis=1)

    X_unique = (
        train.loc[tr_ok, frozen_features]
        .drop_duplicates()
        .astype(float)
        .reset_index(drop=True)
    )

    expected_mean = X_unique[frozen_features].mean(axis=0).to_numpy(dtype=float)
    # StandardScaler uses population variance (ddof=0).
    expected_scale = X_unique[frozen_features].std(axis=0, ddof=0).to_numpy(dtype=float)

    mean_match = np.allclose(scaler.mean_, expected_mean, rtol=1e-12, atol=1e-12)
    scale_match = np.allclose(scaler.scale_, expected_scale, rtol=1e-12, atol=1e-12)

    # 3) Standardized Train fitting matrix should be centered/scaled.
    Xs = scaler.transform(X_unique[frozen_features])
    train_z_mean_max_abs = float(np.max(np.abs(Xs.mean(axis=0))))
    train_z_std_max_abs_error = float(np.max(np.abs(Xs.std(axis=0, ddof=0) - 1.0)))
    standardized_train_ok = (
        train_z_mean_max_abs < 1e-10
        and train_z_std_max_abs_error < 1e-10
    )

    # 4) Manual scaling must equal sklearn transform in the SAME feature order.
    manual = (
        X_unique[frozen_features].to_numpy(dtype=float) - scaler.mean_
    ) / scaler.scale_
    manual_transform_ok = np.allclose(Xs, manual, rtol=1e-12, atol=1e-12)

    # 5) Exact frozen-model prediction reproduction on loan rows.
    out_col = f"{spec_name}_Macro_Cluster_ResolvedBLS"

    train_pred = kmeans.predict(
        scaler.transform(train.loc[tr_ok, frozen_features].astype(float))
    )
    val_pred = kmeans.predict(
        scaler.transform(validation.loc[va_ok, frozen_features].astype(float))
    )

    stored_train = train.loc[tr_ok, out_col].astype(int).to_numpy()
    stored_val = validation.loc[va_ok, out_col].astype(int).to_numpy()

    train_prediction_match = np.array_equal(train_pred, stored_train)
    val_prediction_match = np.array_equal(val_pred, stored_val)

    # Unresolved rows must remain explicit -1, not be fabricated through scaling/imputation.
    train_unresolved_ok = bool(
        (train.loc[~tr_ok, out_col].astype("Int64") == -1).all()
    )
    val_unresolved_ok = bool(
        (validation.loc[~va_ok, out_col].astype("Int64") == -1).all()
    )

    # Non-degenerate Train assignment among complete rows.
    unique_train_labels = np.unique(train_pred)
    nondegenerate_train = len(unique_train_labels) >= 2
    labels_in_range = (
        set(unique_train_labels).issubset(set(range(chosen_k)))
        and set(np.unique(val_pred)).issubset(set(range(chosen_k)))
    )

    # Validation diagnostics only — NOT pass/fail centering requirements.
    Xva_s = scaler.transform(validation.loc[va_ok, frozen_features].astype(float))
    val_z_mean_max_abs = float(np.max(np.abs(Xva_s.mean(axis=0)))) if len(Xva_s) else np.nan
    val_z_std_range = (
        float(Xva_s.std(axis=0, ddof=0).min()) if len(Xva_s) else np.nan,
        float(Xva_s.std(axis=0, ddof=0).max()) if len(Xva_s) else np.nan,
    )

    all_pass = all([
        declared_order_ok,
        scaler_order_ok,
        dimension_ok,
        mean_match,
        scale_match,
        standardized_train_ok,
        manual_transform_ok,
        train_prediction_match,
        val_prediction_match,
        train_unresolved_ok,
        val_unresolved_ok,
        nondegenerate_train,
        labels_in_range,
    ])

    scaling_rows.append({
        "Specification": spec_name,
        "Chosen_K": chosen_k,
        "Frozen_Features": " | ".join(frozen_features),
        "Declared_Order_OK": declared_order_ok,
        "Scaler_Feature_Order_OK": scaler_order_ok,
        "Dimension_OK": dimension_ok,
        "Scaler_Mean_Matches_Unique_Train": mean_match,
        "Scaler_Scale_Matches_Unique_Train": scale_match,
        "Train_Z_Mean_MaxAbs": train_z_mean_max_abs,
        "Train_Z_Std_MaxAbsError": train_z_std_max_abs_error,
        "Train_Standardization_OK": standardized_train_ok,
        "Manual_vs_Sklearn_Transform_OK": manual_transform_ok,
        "Train_Prediction_Reproduced": train_prediction_match,
        "Validation_Prediction_Reproduced": val_prediction_match,
        "Train_Unresolved_is_minus1": train_unresolved_ok,
        "Validation_Unresolved_is_minus1": val_unresolved_ok,
        "Nondegenerate_Train_Assignment": nondegenerate_train,
        "Labels_In_Range": labels_in_range,
        "Validation_Z_Mean_MaxAbs_DIAGNOSTIC_ONLY": val_z_mean_max_abs,
        "Validation_Z_Std_Min_DIAGNOSTIC_ONLY": val_z_std_range[0],
        "Validation_Z_Std_Max_DIAGNOSTIC_ONLY": val_z_std_range[1],
        "PASS": all_pass,
    })

scaling_audit = pd.DataFrame(scaling_rows)
display(scaling_audit)

scaling_audit.to_csv(
    QA_DIR / "06C_corrected_scaling_structure_audit.csv",
    index=False,
)

assert scaling_audit["PASS"].all(), (
    "Corrected macro scaling/frozen-transform audit failed. "
    "The production parquet will NOT be overwritten."
)

# 8B. Confirm Regular and LS/Strict models are genuinely separate fitted objects.
pair_rows = []
for geography in ["State", "MSA"]:
    for profile in ["A", "B"]:
        reg_spec = f"Regular_{geography}_{profile}"
        ls_spec = f"LS_{geography}_{profile}"

        reg_key = [k for k in model_store if k[0] == reg_spec][0]
        ls_key = [k for k in model_store if k[0] == ls_spec][0]

        reg_bundle = model_store[reg_key]
        ls_bundle = model_store[ls_key]

        pair_rows.append({
            "Geography": geography,
            "Profile": profile,
            "Separate_Scaler_Objects": reg_bundle["scaler"] is not ls_bundle["scaler"],
            "Separate_KMeans_Objects": reg_bundle["kmeans"] is not ls_bundle["kmeans"],
        })

pit_separation_audit = pd.DataFrame(pair_rows)
pit_separation_audit["PASS"] = (
    pit_separation_audit["Separate_Scaler_Objects"]
    & pit_separation_audit["Separate_KMeans_Objects"]
)

display(pit_separation_audit)
pit_separation_audit.to_csv(
    QA_DIR / "06C_regular_vs_ls_separate_fit_audit.csv",
    index=False,
)
assert pit_separation_audit["PASS"].all()

print("=" * 100)
print("CORRECTED SCALING STRUCTURE: PASS")
print("=" * 100)
print("Scaler fit source        : unique complete TRAIN macro profiles only")
print("Feature order source     : bundle['features'] / MACRO_SPECS declared order")
print("Validation refit         : NO")
print("Validation re-centering  : NO")
print("Frozen prediction replay : PASS")
print("Regular vs LS fit        : SEPARATE")
print("=" * 100)


,Specification,Chosen_K,Frozen_Features,Declared_Order_OK,Scaler_Feature_Order_OK,Dimension_OK,Scaler_Mean_Matches_Unique_Train,Scaler_Scale_Matches_Unique_Train,Train_Z_Mean_MaxAbs,Train_Z_Std_MaxAbsError,Train_Standardization_OK,Manual_vs_Sklearn_Transform_OK,Train_Prediction_Reproduced,Validation_Prediction_Reproduced,Train_Unresolved_is_minus1,Validation_Unresolved_is_minus1,Nondegenerate_Train_Assignment,Labels_In_Range,Validation_Z_Mean_MaxAbs_DIAGNOSTIC_ONLY,Validation_Z_Std_Min_DIAGNOSTIC_ONLY,Validation_Z_Std_Max_DIAGNOSTIC_ONLY,PASS
0,Regular_State_A,2,FHFA_State_HPI | FHFA_State_HPI_Growth_12M | B...,True,True,True,True,True,3.981528e-16,0.000000e+00,True,True,True,True,True,True,True,True,0.712934,0.554878,1.195923,True
1,Regular_State_B,2,FHFA_State_HPI_Growth_12M | BLS_Unemployment_R...,True,True,True,True,True,3.981528e-16,0.000000e+00,True,True,True,True,True,True,True,True,0.712934,0.554878,0.729793,True
2,LS_State_A,2,LS_FHFA_State_HPI | LS_FHFA_State_HPI_Growth_1...,True,True,True,True,True,7.963554e-16,1.110223e-16,True,True,True,True,True,True,True,True,0.713657,0.559482,1.194884,True
3,LS_State_B,2,LS_FHFA_State_HPI_Growth_12M | LS_BLS_Unemploy...,True,True,True,True,True,8.532379e-17,1.110223e-16,True,True,True,True,True,True,True,True,0.713657,0.559482,0.695679,True
4,Regular_MSA_A,2,FHFA_MSA_HPI | FHFA_MSA_HPI_Growth_12M | BLS_U...,True,True,True,True,True,1.571828e-16,0.000000e+00,True,True,True,True,True,True,True,True,1.212075,0.551922,1.442368,True
5,Regular_MSA_B,2,FHFA_MSA_HPI_Growth_12M | BLS_Unemployment_Rat...,True,True,True,True,True,1.397181e-16,0.000000e+00,True,True,True,True,True,True,True,True,0.707024,0.551922,0.700316,True
6,LS_MSA_A,2,LS_FHFA_MSA_HPI | LS_FHFA_MSA_HPI_Growth_12M |...,True,True,True,True,True,1.222627e-16,0.000000e+00,True,True,True,True,True,True,True,True,1.206489,0.557140,1.446671,True
7,LS_MSA_B,2,LS_FHFA_MSA_HPI_Growth_12M | LS_BLS_Unemployme...,True,True,True,True,True,7.423092e-17,0.000000e+00,True,True,True,True,True,True,True,True,0.709089,0.557140,0.665824,True


,Geography,Profile,Separate_Scaler_Objects,Separate_KMeans_Objects,PASS
0,State,A,True,True,True
1,State,B,True,True,True
2,MSA,A,True,True,True
3,MSA,B,True,True,True


CORRECTED SCALING STRUCTURE: PASS
Scaler fit source        : unique complete TRAIN macro profiles only
Feature order source     : bundle['features'] / MACRO_SPECS declared order
Validation refit         : NO
Validation re-centering  : NO
Frozen prediction replay : PASS
Regular vs LS fit        : SEPARATE


## 9. Save → reload → persisted-artifact verification

The canonical parquet is overwritten **only after** all mandatory FE and corrected scaling audits pass.  
After writing, the notebook reloads the parquet and verifies row/column identity, `Loan_ID` order, target preservation, engineered-column persistence, and macro-cluster persistence.

In [14]:
# Convert categorical columns to string before parquet if the local parquet engine has categorical metadata issues.
# Here we preserve pandas categoricals unless save fails.
try:
    fe.to_parquet(OUTPUT_PATH, index=False)
except Exception as e:
    warnings.warn(f"Categorical parquet save failed ({e}); converting categorical columns to string and retrying.")
    fe_save = fe.copy()
    for c in fe_save.select_dtypes(include="category").columns:
        fe_save[c] = fe_save[c].astype("string")
    fe_save.to_parquet(OUTPUT_PATH, index=False)
    fe = fe_save

reload = pd.read_parquet(OUTPUT_PATH)
post = []
def postcheck(name, passed, detail=""):
    post.append({"Check":name,"Passed":bool(passed),"Detail":str(detail)})
postcheck("Reload row count", len(reload)==len(fe), f"{len(reload):,}")
postcheck("Reload column count", reload.shape[1]==fe.shape[1], f"{reload.shape[1]}")
postcheck("Reload Loan_ID order", reload["Loan_ID"].reset_index(drop=True).equals(fe["Loan_ID"].reset_index(drop=True)))
postcheck("Reload Target_24M", reload["Target_24M"].reset_index(drop=True).equals(fe["Target_24M"].reset_index(drop=True)))
postcheck("All engineered columns persisted", set(new_cols).issubset(reload.columns), f"N engineered={len(new_cols)}")

# Re-check every production macro cluster column after parquet reload.
for spec_name in MACRO_SPECS:
    out_col = f"{spec_name}_Macro_Cluster_ResolvedBLS"
    postcheck(
        f"Reload {out_col} exact",
        reload[out_col].astype("Int64").reset_index(drop=True).equals(
            fe[out_col].astype("Int64").reset_index(drop=True)
        ),
    )
post_table = pd.DataFrame(post)
display(post_table)
post_table.to_csv(QA_DIR / "06C_post_save_validation.csv", index=False)
assert post_table["Passed"].all(), "Post-save validation failed."
print("Saved and reloaded successfully:", OUTPUT_PATH)

,Check,Passed,Detail
0,Reload row count,True,"209,041"
1,Reload column count,True,91
2,Reload Loan_ID order,True,
3,Reload Target_24M,True,
4,All engineered columns persisted,True,N engineered=25
5,Reload Regular_State_A_Macro_Cluster_ResolvedB...,True,
6,Reload Regular_State_B_Macro_Cluster_ResolvedB...,True,
7,Reload LS_State_A_Macro_Cluster_ResolvedBLS exact,True,
8,Reload LS_State_B_Macro_Cluster_ResolvedBLS exact,True,
9,Reload Regular_MSA_A_Macro_Cluster_ResolvedBLS...,True,


Saved and reloaded successfully: C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_feature_engineered_train_validation.parquet


## 10. Final corrected 06C audit summary

A final PASS means the regenerated parquet is internally consistent with the frozen feature definitions, Train-only learning contract, corrected macro feature ordering, and scaler/K-means structure.

In [15]:
summary = {
    "input_file": str(INPUT_PATH),
    "output_file": str(OUTPUT_PATH),
    "rows": int(len(reload)),
    "columns_before": int(df0.shape[1]),
    "columns_after": int(reload.shape[1]),
    "train_rows": int((reload["Split"].astype("string")=="Train").sum()),
    "validation_rows": int((reload["Split"].astype("string")=="Validation").sum()),
    "spatial_k": SPATIAL_K,
    "macro_candidate_specs": list(MACRO_SPECS.keys()),
    "macro_labor_semantics": "retained resolved/final PIT BLS fields; provenance audited; not claimed native State/MSA at both levels",
    "regular_and_ls_fitted_separately": True,
    "validation_used_for_fit": False,
    "mandatory_qa_passed": bool(qa_table["Passed"].all()),
    "post_save_qa_passed": bool(post_table["Passed"].all()),
    "corrected_scaling_structure_passed": bool(scaling_audit["PASS"].all()),
    "regular_vs_ls_separate_fit_audit_passed": bool(pit_separation_audit["PASS"].all()),
    "macro_scaler_fit_population": "unique complete Train macro profiles only",
    "macro_feature_order_source": "per-specification ordered bundle['features']",
}
with open(QA_DIR / "06C_final_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("="*100)
print("06C FEATURE ENGINEERING COMPLETE")
print("="*100)
for k,v in summary.items(): print(f"{k}: {v}")
print("\nDecision status: Profile A and B are BOTH retained as candidate macro specifications.")
print("Regular PIT and LS/Strict PIT were fit as SEPARATE FE pipelines.")
print("Corrected macro scaling structure audit: PASS")
print("Canonical parquet has been regenerated from the 06B master and verified after reload.")


06C FEATURE ENGINEERING COMPLETE
input_file: C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_feature_engineering_master_train_validation.parquet
output_file: C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_feature_engineered_train_validation.parquet
rows: 209041
columns_before: 66
columns_after: 91
train_rows: 160003
validation_rows: 49038
spatial_k: 8
macro_candidate_specs: ['Regular_State_A', 'Regular_State_B', 'LS_State_A', 'LS_State_B', 'Regular_MSA_A', 'Regular_MSA_B', 'LS_MSA_A', 'LS_MSA_B']
macro_labor_semantics: retained resolved/final PIT BLS fields; provenance audited; not claimed native State/MSA at both levels
regular_and_ls_fitted_separately: True
validation_used_for_fit: False
mandatory_qa_passed: True
post_save_qa_passed: True
corrected_scaling_structure_passed: True
regular_vs_ls_separate_fit_audit_passed: Tr

# 06C — Final Production Validation Record

## Final Status

\[
\boxed{\textbf{06C Feature Engineering = PASS / FREEZE}}
\]

The corrected 06C pipeline successfully regenerated and validated the production feature-engineered dataset:

`SFLLD_feature_engineered_train_validation.parquet`

The final pipeline follows the required leakage-control principle:

\[
\boxed{
\text{Train learns}
\rightarrow
\text{Freeze}
\rightarrow
\text{Validation applies}
}
\]

Validation observations were not used to fit the macro scalers, K-Means models, centroids, or other learned FE parameters.

---

## 1. Final Dataset Integrity

| Item | Final Result |
|---|---:|
| Input columns | 66 |
| Engineered columns added | 25 |
| Final columns | **91** |
| Total rows | **209,041** |
| Train rows | **160,003** |
| Validation rows | **49,038** |
| Row count preserved | PASS |
| `Loan_ID` order preserved | PASS |
| `Target_24M` preserved | PASS |
| Engineered features persisted after reload | PASS |
| Mandatory pre-save QA | **PASS** |
| Post-save QA | **PASS** |

The final parquet was reloaded after writing and the persisted artifact was independently checked. All eight macro-cluster columns reproduced the in-memory results exactly.

---

## 2. Corrected Macro Scaling Structure

The final implementation uses an independent frozen feature order for every macro specification.

For specification \(s\),

\[
\mathbf{x}^{(s)}
=
[x_1,x_2,\ldots,x_p]
\]

is stored explicitly in:

`bundle["features"]`

and the Train-only scaler is fitted as

\[
z_j
=
\frac{x_j-\mu_{j,\mathrm{Train}}}
{\sigma_{j,\mathrm{Train}}}.
\]

The same frozen parameters are then applied to Validation:

\[
z_{j,\mathrm{Val}}
=
\frac{x_{j,\mathrm{Val}}-\mu_{j,\mathrm{Train}}}
{\sigma_{j,\mathrm{Train}}}.
\]

No Validation re-fitting or re-centering is permitted.

### Scaling QA

The following checks passed for all eight macro specifications:

- declared feature order = frozen bundle feature order;
- scaler feature order is correct;
- scaler dimensionality matches the macro vector;
- `scaler.mean_` matches the unique complete Train profiles;
- `scaler.scale_` matches the unique complete Train profiles;
- standardized Train fitting profiles have approximately zero mean;
- standardized Train fitting profiles have approximately unit population variance;
- manual \((X-\mu)/\sigma\) calculation matches `StandardScaler.transform()`;
- frozen Train cluster assignments are exactly reproducible;
- frozen Validation cluster assignments are exactly reproducible;
- unresolved macro profiles remain explicitly coded as `-1`;
- Train cluster assignments are non-degenerate;
- cluster labels remain within the valid fitted range.

Therefore,

\[
\boxed{\text{Corrected Scaling Structure = PASS}}
\]

Regular PIT and LS/Strict PIT were also verified to use **separate scaler and K-Means objects**.

---

## 3. MSA Macro Coverage — Structural Unresolved Observations

MSA macro coverage is intentionally lower than State macro coverage.

| Geography | Train Complete | Validation Complete | Train Unresolved | Validation Unresolved |
|---|---:|---:|---:|---:|
| State | ~99.95% | ~99.97% | 86 | 13 |
| MSA | ~80.12% | ~82.49% | **31,806** | **8,588** |

The unresolved MSA observations originate from incomplete underlying MSA macro vectors.

They are **not** median-imputed and are **not** assigned artificially to one of the learned economic regimes.

Instead,

\[
\text{Incomplete MSA Macro Vector}
\Rightarrow
\boxed{\text{Macro Cluster}=-1}.
\]

This is an intentional structural-missingness treatment rather than a pipeline error.

Therefore:

\[
\boxed{
\text{MSA unresolved}=-1
\text{ is retained as an explicit state}
}
\]

rather than manufacturing a macroeconomic regime for observations whose complete MSA information is unavailable.

---

## 4. Validation Standardization Shift

Validation is **not expected** to have standardized mean \(0\) and standard deviation \(1\).

Only Train defines:

\[
\mu_{\mathrm{Train}},
\qquad
\sigma_{\mathrm{Train}}.
\]

Consequently,

\[
E[Z_{\mathrm{Train}}]\approx0,
\]

but generally,

\[
E[Z_{\mathrm{Validation}}]\neq0.
\]

For example, `Regular_MSA_A` exhibits a Validation standardized mean displacement with approximately:

\[
Z_{\text{Mean, MaxAbs}}\approx1.212.
\]

The corrected frozen-transform audit demonstrates that this is **not caused by incorrect feature ordering or incorrect scaling**.

It represents genuine Train-to-Validation macroeconomic distribution shift.

Therefore:

\[
\boxed{
\text{Validation distribution shift}
\neq
\text{scaling bug}
}
\]

and the shift should be retained as a temporal/distribution-shift diagnostic rather than removed through Validation re-standardization.

---

## 5. Macro \(K\) Selection Record

For each macro specification, candidate values

\[
\boxed{K\in\{2,3,4,5,6\}}
\]

were evaluated using Train-only macro profiles.

The production selection rule retained from 06C is:

\[
\boxed{
K^*
=
\arg\max_K
\operatorname{Silhouette}(K)
}
\]

with the smaller \(K\) preferred in the event of a tie.

The corrected production run again selected:

\[
\boxed{K=2}
\]

for all eight macro specifications.

This result was **not hard-coded**. All candidate values were evaluated.

For example:

### `Regular_MSA_A`

| K | Silhouette |
|---:|---:|
| **2** | **0.2397** |
| 3 | 0.2025 |
| 4 | 0.1922 |
| 5 | 0.1838 |
| 6 | 0.1903 |

### `LS_MSA_B`

| K | Silhouette |
|---:|---:|
| **2** | **0.2498** |
| 3 | 0.1989 |
| 4 | 0.2169 |
| 5 | 0.2100 |
| 6 | 0.2138 |

The separate 06C robustness/economic-interpretation audit evaluated the richer \(K\) alternatives using additional clustering diagnostics and economic interpretation.

Therefore, the production \(K=2\) result is retained together with the robustness audit as supporting methodological documentation.

---

## 6. Regular PIT vs. LS/Strict PIT

Regular PIT and LS/Strict PIT remain separate FE pipelines.

For each geography/profile combination,

\[
\text{Regular}
\neq
\text{LS/Strict}
\]

in terms of fitted scaler and fitted K-Means objects.

The following specifications remain independently constructed:

- `Regular_State_A`
- `Regular_State_B`
- `Regular_MSA_A`
- `Regular_MSA_B`
- `LS_State_A`
- `LS_State_B`
- `LS_MSA_A`
- `LS_MSA_B`

This preserves the intended comparison between the Regular PIT and Strict PIT information sets.

---

## 7. Final Production Decision

All mandatory FE integrity checks, corrected scaling checks, frozen-model reproduction checks, Regular-vs-LS separation checks, and post-save artifact checks passed.

The final production dataset is therefore:

`SFLLD_feature_engineered_train_validation.parquet`

with:

\[
\boxed{
209{,}041\text{ rows}
\times
91\text{ columns}
}
\]

and

\[
\boxed{
160{,}003\text{ Train}
+
49{,}038\text{ Validation}
}
\]

The corrected production run records:

- `validation_used_for_fit = False`
- `mandatory_qa_passed = True`
- `post_save_qa_passed = True`
- `corrected_scaling_structure_passed = True`
- `regular_vs_ls_separate_fit_audit_passed = True`

### Final Decision

\[
\boxed{
\textbf{06C = PASS / FREEZE}
}
\]

No further modification to the 06C production feature-engineering pipeline is required before proceeding to the next modeling-data preparation stage.